# LeetCode 1669 — Merge In Between Linked Lists bằng Python

Link đề: https://leetcode.com/problems/merge-in-between-linked-lists/ · Độ khó: **Medium** · Chủ đề: Linked List

Skeleton LeetCode:

```python
# Definition for singly-linked list.
# class ListNode:
#     def __init__(self, val=0, next=None):
#         self.val = val
#         self.next = next
class Solution:
    def mergeInBetween(self, list1: ListNode, a: int, b: int, list2: ListNode) -> ListNode:
        ...
```

Bài này là bài tập "nối dây" kinh điển trên danh sách liên kết: tìm đúng **hai điểm cắt** trên `list1`, tìm **đuôi** của `list2`, rồi nối lại ba con trỏ.

## 1. Mục tiêu học

1. Nắm cấu trúc **danh sách liên kết đơn** (`ListNode`) và cách dựng/xuất nó từ list Python để chạy thử như LeetCode.
2. Biết cách đi tới **nút thứ `k`** (0-indexed) và đếm đúng số bước — tránh lỗi lệch một (off-by-one).
3. Hiểu rằng bài chỉ cần sửa **hai con trỏ `next`**: nút `a−1` → đầu `list2`, đuôi `list2` → nút `b+1`.
4. Đi từ cách chuyển sang list Python (`O(n+m)` bộ nhớ) → hai con trỏ `O(1)` bộ nhớ → một lượt duyệt.
5. Kiểm thử bằng cách đối chiếu với phép cắt-ghép list Python `arr1[:a] + arr2 + arr1[b+1:]` trên dữ liệu ngẫu nhiên.

## 2. Đề bài & ví dụ

Cho hai danh sách liên kết `list1` (độ dài `n`) và `list2` (độ dài `m`).

Hãy **xóa** các nút của `list1` từ nút thứ `a` tới nút thứ `b` (đánh số từ 0, **bao gồm** cả hai đầu) và **đặt toàn bộ `list2` vào chỗ đó**. Trả về đầu của danh sách kết quả.

**Ví dụ 1:** `list1 = [10,1,13,6,9,5]`, `a = 3`, `b = 4`, `list2 = [1000000,1000001,1000002]` → `[10,1,13,1000000,1000001,1000002,5]` (xóa nút chỉ số 3 và 4, tức `6` và `9`).

**Ví dụ 2:** `list1 = [0,1,2,3,4,5,6]`, `a = 2`, `b = 5`, `list2 = [1000000,1000001,1000002,1000003,1000004]` → `[0,1,1000000,1000001,1000002,1000003,1000004,6]`.

**Ràng buộc:**
- `3 ≤ list1.length ≤ 10⁴`
- `1 ≤ a ≤ b < list1.length − 1` ⇒ **không bao giờ** xóa nút đầu hay nút cuối của `list1`.
- `1 ≤ list2.length ≤ 10⁴`

In [1]:
import random
import time
from typing import Optional, List

class ListNode:
    def __init__(self, val=0, next=None):
        self.val = val
        self.next = next
    def __repr__(self):
        return f"ListNode({self.val})"

def build_list(arr):
    """Dựng linked list từ list Python, trả về nút đầu (None nếu rỗng)."""
    dummy = ListNode()
    cur = dummy
    for v in arr:
        cur.next = ListNode(v)
        cur = cur.next
    return dummy.next

def to_list(head, limit=10**6):
    """Xuất linked list ra list Python (limit chống vòng lặp vô hạn khi có chu trình)."""
    out = []
    while head is not None and len(out) < limit:
        out.append(head.val)
        head = head.next
    return out

examples = [
    ([10, 1, 13, 6, 9, 5], 3, 4, [1000000, 1000001, 1000002], [10, 1, 13, 1000000, 1000001, 1000002, 5]),
    ([0, 1, 2, 3, 4, 5, 6], 2, 5, [1000000, 1000001, 1000002, 1000003, 1000004],
     [0, 1, 1000000, 1000001, 1000002, 1000003, 1000004, 6]),
]
for l1, a, b, l2, out in examples:
    print("list1 =", to_list(build_list(l1)), f"a={a}, b={b}", "list2 =", l2, "-> kỳ vọng", out)

list1 = [10, 1, 13, 6, 9, 5] a=3, b=4 list2 = [1000000, 1000001, 1000002] -> kỳ vọng [10, 1, 13, 1000000, 1000001, 1000002, 5]
list1 = [0, 1, 2, 3, 4, 5, 6] a=2, b=5 list2 = [1000000, 1000001, 1000002, 1000003, 1000004] -> kỳ vọng [0, 1, 1000000, 1000001, 1000002, 1000003, 1000004, 6]


## 3. Kiến thức nền

### 3.1 Nút danh sách liên kết và "dummy head"

Mỗi `ListNode` có `val` và `next`. Khi dựng list, dùng một nút giả `dummy` đứng trước đầu thật giúp không phải xử lý riêng trường hợp "list đang rỗng". Đầu thật là `dummy.next`.

In [2]:
head = build_list([1, 2, 3])
print(head, "->", head.next, "->", head.next.next, "->", head.next.next.next)
print("to_list:", to_list(head))

ListNode(1) -> ListNode(2) -> ListNode(3) -> None
to_list: [1, 2, 3]


### 3.2 Đi tới nút thứ `k` (0-indexed)

Bắt đầu ở `head` (chỉ số 0), mỗi lần `cur = cur.next` tăng chỉ số lên 1. Muốn tới chỉ số `k` thì đi **đúng `k` bước**. Đây là chỗ hay lệch một nhất — hãy luôn tự hỏi "sau vòng lặp, `cur` đang đứng ở chỉ số mấy?".

In [3]:
def kth(head, k):
    cur = head
    for _ in range(k):
        cur = cur.next
    return cur

head = build_list([10, 1, 13, 6, 9, 5])
for k in range(6):
    print(f"kth({k}) =", kth(head, k).val)

kth(0) = 10
kth(1) = 1
kth(2) = 13
kth(3) = 6
kth(4) = 9
kth(5) = 5


### 3.3 Tìm nút đuôi

Đi tới khi `cur.next is None`. Chi phí `O(độ dài)`.

In [4]:
def tail(head):
    cur = head
    while cur.next is not None:
        cur = cur.next
    return cur

print("đuôi của [7, 8, 9]:", tail(build_list([7, 8, 9])).val)

đuôi của [7, 8, 9]: 9


### 3.4 Nối lại con trỏ (splice)

Nối `X → Y` chỉ là một phép gán `X.next = Y`. Điều quan trọng là **giữ được tham chiếu** tới mọi nút cần dùng **trước** khi ghi đè `next` — nếu không, phần phía sau sẽ "mất dấu" (vẫn tồn tại trong bộ nhớ nhưng ta không còn đường tới nó).

In [5]:
x = build_list([1, 2, 3, 4])
second = x.next
after = second.next.next          # giữ tham chiếu tới nút 4 TRƯỚC khi cắt
second.next = after               # bỏ qua nút 3
print("sau khi bỏ nút 3:", to_list(x))

sau khi bỏ nút 3: [1, 2, 4]


### 3.5 Cắt-ghép list Python

`arr[:a] + arr2 + arr[b+1:]` chính là đáp án mong muốn ở dạng list Python — dùng làm **chuẩn** (oracle) để kiểm thử.

In [6]:
arr1, arr2 = [10, 1, 13, 6, 9, 5], [1000000, 1000001, 1000002]
a, b = 3, 4
print("giữ đầu   arr1[:a]   =", arr1[:a])
print("chèn      arr2       =", arr2)
print("giữ cuối  arr1[b+1:] =", arr1[b + 1:])
print("kết quả:", arr1[:a] + arr2 + arr1[b + 1:])

giữ đầu   arr1[:a]   = [10, 1, 13]
chèn      arr2       = [1000000, 1000001, 1000002]
giữ cuối  arr1[b+1:] = [5]
kết quả: [10, 1, 13, 1000000, 1000001, 1000002, 5]


## 4. Ý tưởng nền tảng / trực giác — tính bằng tay

Ví dụ 1: `list1 = 10 → 1 → 13 → 6 → 9 → 5`, `a = 3`, `b = 4`.

| chỉ số | 0 | 1 | 2 | 3 | 4 | 5 |
|---|---|---|---|---|---|---|
| giá trị | 10 | 1 | **13** | ~~6~~ | ~~9~~ | **5** |
| vai trò | | | `before` (chỉ số `a−1 = 2`) | xóa | xóa | `after` (chỉ số `b+1 = 5`) |

Chỉ cần:
1. `before.next = list2` (13 → 1000000)
2. `tail(list2).next = after` (1000002 → 5)

```
10 → 1 → 13 ─┐            ┌─→ 5
             └→ 1000000 → 1000001 → 1000002
```

Vì `a ≥ 1` nên `before` luôn tồn tại; vì `b < n − 1` nên `after` luôn tồn tại — đề đã loại bỏ hai trường hợp biên khó.

In [7]:
head = build_list([10, 1, 13, 6, 9, 5]); a, b = 3, 4
before, after = kth(head, a - 1), kth(head, b + 1)
print("before =", before.val, "(chỉ số", a - 1, ") | after =", after.val, "(chỉ số", b + 1, ")")
removed = []
cur = before.next
while cur is not after:
    removed.append(cur.val); cur = cur.next
print("các nút bị xóa:", removed)

before = 13 (chỉ số 2 ) | after = 5 (chỉ số 5 )
các nút bị xóa: [6, 9]


## 5. Cách 1 — Baseline: chuyển sang list Python rồi dựng lại

**Ý tưởng:** không động vào con trỏ. Đổi cả hai danh sách liên kết sang list Python, cắt-ghép bằng slicing, rồi dựng danh sách liên kết **mới**.

- Đơn giản, khó sai.
- Tốn `O(n + m)` bộ nhớ phụ và tạo **nút mới** — trên LeetCode vẫn được chấp nhận (họ chỉ so giá trị), nhưng phỏng vấn thường yêu cầu **sửa tại chỗ**.

In [8]:
def merge_baseline(list1, a, b, list2):
    arr1, arr2 = to_list(list1), to_list(list2)
    return build_list(arr1[:a] + arr2 + arr1[b + 1:])

for l1, a, b, l2, out in examples:
    got = to_list(merge_baseline(build_list(l1), a, b, build_list(l2)))
    print(got)
    assert got == out

[10, 1, 13, 1000000, 1000001, 1000002, 5]
[0, 1, 1000000, 1000001, 1000002, 1000003, 1000004, 6]


**Độ phức tạp Cách 1:** Time `O(n + m)`, Space `O(n + m)` (hai list Python + `n − (b−a+1) + m` nút mới).

**Nút thắt:** tạo lại toàn bộ danh sách dù chỉ cần đổi **2 con trỏ**. Hơn nữa, các nút của `list2` trong kết quả **không phải** là các nút gốc — nếu đề (hoặc người phỏng vấn) kiểm tra danh tính nút thì cách này sai:

In [9]:
l1, l2 = build_list([10, 1, 13, 6, 9, 5]), build_list([7, 8])
res = merge_baseline(l1, 3, 4, l2)
print("đầu kết quả có phải là đầu list1 gốc?", res is l1)
print("nút 7 trong kết quả có phải nút 7 của list2 gốc?", kth(res, 3) is l2)

đầu kết quả có phải là đầu list1 gốc? False
nút 7 trong kết quả có phải nút 7 của list2 gốc? False


## 6. Ý tưởng thuật toán / bước đột phá

**Insight:** kết quả = `list1[0..a−1]` + `list2` + `list1[b+1..n−1]`. Ba đoạn này **đã nằm sẵn** trong bộ nhớ, đã nối liền bên trong; ta chỉ cần nối **hai mối nối** giữa chúng:

- **Mối nối 1:** nút chỉ số `a−1` của `list1` → đầu `list2`.
- **Mối nối 2:** đuôi `list2` → nút chỉ số `b+1` của `list1`.

**Quy tắc "nếu… thì…":**
- Nếu đứng ở `head` và đi `a−1` bước **thì** tới `before`.
- Nếu đứng ở `head` và đi `b+1` bước **thì** tới `after` (hoặc từ `before` đi thêm `b − a + 2` bước).
- **Phải tìm `after` trước khi** gán `before.next = list2` — nếu không, đường đi từ `before` sang phần sau đã bị thay bằng `list2`.

**Thứ tự an toàn:** tìm `before`, tìm `after`, tìm `tail2` → gán hai con trỏ. (Tùy chọn: cắt `b.next = None` để đoạn bị xóa không còn trỏ vào list kết quả.)

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Hai con trỏ `before` / `after` (sửa tại chỗ, `O(1)` bộ nhớ)

**Từng đoạn:**
1. `before = head`, đi `a − 1` bước.
2. `after = before`, đi thêm `b − a + 2` bước (tới chỉ số `b + 1`).
3. `tail2 = list2`, đi tới cuối.
4. `before.next = list2`, `tail2.next = after`.

Code demo:

In [10]:
def demo_two_pointers(list1, a, b, list2):
    before = list1
    for _ in range(a - 1):
        before = before.next
    after = before
    for _ in range(b - a + 2):
        after = after.next
    tail2 = list2
    steps2 = 0
    while tail2.next:
        tail2 = tail2.next; steps2 += 1
    print(f"before={before.val}, after={after.val}, tail2={tail2.val} (đi {steps2} bước trên list2)")
    before.next = list2
    tail2.next = after
    return list1

print(to_list(demo_two_pointers(build_list([0, 1, 2, 3, 4, 5, 6]), 2, 5, build_list([100, 101, 102]))))

before=1, after=6, tail2=102 (đi 2 bước trên list2)
[0, 1, 100, 101, 102, 6]


Code nộp LeetCode của Cách 2:

In [11]:
class SolutionTwoPointers:
    def mergeInBetween(self, list1: ListNode, a: int, b: int, list2: ListNode) -> ListNode:
        before = list1
        for _ in range(a - 1):            # tới chỉ số a-1
            before = before.next
        after = before
        for _ in range(b - a + 2):        # tới chỉ số b+1
            after = after.next
        tail2 = list2
        while tail2.next:                 # đuôi của list2
            tail2 = tail2.next
        before.next = list2
        tail2.next = after
        return list1

for l1, a, b, l2, out in examples:
    assert to_list(SolutionTwoPointers().mergeInBetween(build_list(l1), a, b, build_list(l2))) == out
print("Cách 2 qua 2 ví dụ")

Cách 2 qua 2 ví dụ


**Độ phức tạp Cách 2:** Time `O(b + m)` (đi `b+1` bước trên `list1`, `m` bước trên `list2`) ≤ `O(n + m)`; Space `O(1)`.

### 7.2 Cách 3 — Một lượt duyệt với chỉ số

Thay vì hai vòng `for` đếm bước, duyệt `list1` **một lần** kèm biến chỉ số `i`; khi `i == a − 1` ghi nhớ `before`, khi `i == b + 1` ghi nhớ `after` và dừng sớm. Cách này dễ đọc khi bài có nhiều mốc cần đánh dấu.

Code demo:

In [12]:
def demo_one_pass(list1, a, b, list2):
    before = after = None
    i, cur = 0, list1
    while cur is not None:
        if i == a - 1:
            before = cur; print(f"  i={i}: ghi nhớ before = {cur.val}")
        if i == b + 1:
            after = cur; print(f"  i={i}: ghi nhớ after  = {cur.val} -> dừng"); break
        cur = cur.next; i += 1
    tail2 = list2
    while tail2.next:
        tail2 = tail2.next
    before.next, tail2.next = list2, after
    return list1

print(to_list(demo_one_pass(build_list([10, 1, 13, 6, 9, 5]), 3, 4, build_list([1000000, 1000001, 1000002]))))

  i=2: ghi nhớ before = 13
  i=5: ghi nhớ after  = 5 -> dừng
[10, 1, 13, 1000000, 1000001, 1000002, 5]


Code nộp LeetCode của Cách 3:

In [13]:
class SolutionOnePass:
    def mergeInBetween(self, list1: ListNode, a: int, b: int, list2: ListNode) -> ListNode:
        before = after = None
        cur, i = list1, 0
        while cur:
            if i == a - 1:
                before = cur
            if i == b + 1:
                after = cur
                break
            cur = cur.next
            i += 1
        tail2 = list2
        while tail2.next:
            tail2 = tail2.next
        before.next = list2
        tail2.next = after
        return list1

for l1, a, b, l2, out in examples:
    assert to_list(SolutionOnePass().mergeInBetween(build_list(l1), a, b, build_list(l2))) == out
print("Cách 3 qua 2 ví dụ")

Cách 3 qua 2 ví dụ


**Độ phức tạp Cách 3:** Time `O(b + m)`, Space `O(1)`.

### 7.3 Cách 4 — Dummy node tổng quát (xử lý được cả `a = 0` hoặc `b = n − 1`)

Đề đảm bảo `1 ≤ a` và `b < n − 1`, nhưng trong thực tế (và các biến thể) ta hay phải xử lý cả khi xóa **từ đầu** hoặc **tới cuối**. Đặt `dummy → list1`; khi đó "nút trước chỉ số `a`" là nút ở vị trí `a` tính từ `dummy` — luôn tồn tại. `after` có thể là `None` (khi `b = n − 1`), vẫn gán được.

In [14]:
class SolutionDummy:
    def mergeInBetween(self, list1: ListNode, a: int, b: int, list2: ListNode) -> ListNode:
        dummy = ListNode(0, list1)
        before = dummy
        for _ in range(a):                 # dummy là "chỉ số -1" => a bước tới chỉ số a-1
            before = before.next
        after = before.next
        for _ in range(b - a + 1):         # bỏ qua b-a+1 nút bị xóa
            after = after.next             # có thể thành None nếu b = n-1
        if list2 is None:                  # tổng quát: list2 có thể rỗng
            before.next = after
            return dummy.next
        tail2 = list2
        while tail2.next:
            tail2 = tail2.next
        before.next = list2
        tail2.next = after
        return dummy.next

for l1, a, b, l2, out in examples:
    assert to_list(SolutionDummy().mergeInBetween(build_list(l1), a, b, build_list(l2))) == out
# ngoài ràng buộc đề (tự dựng): xóa từ đầu / tới cuối
print(to_list(SolutionDummy().mergeInBetween(build_list([1, 2, 3, 4]), 0, 1, build_list([9]))))
print(to_list(SolutionDummy().mergeInBetween(build_list([1, 2, 3, 4]), 2, 3, build_list([9, 8]))))

[9, 3, 4]
[1, 2, 9, 8]


**Độ phức tạp Cách 4:** Time `O(b + m)`, Space `O(1)`.

## 8. TRACE từng bước

Trace Cách 2 trên ví dụ 2: `list1 = [0,1,2,3,4,5,6]`, `a = 2`, `b = 5`, `list2` có 5 nút.

| giai đoạn | bước | con trỏ | đang ở chỉ số | giá trị |
|---|---|---|---|---|
| tìm before | 0 | before | 0 | 0 |
| | 1 | before | 1 | 1 ← dừng (a−1 = 1) |
| tìm after (b−a+2 = 5 bước) | 1..5 | after | 2, 3, 4, 5, 6 | 6 ← dừng (b+1 = 6) |
| tìm tail2 | 4 bước | tail2 | 4 (của list2) | 1000004 |
| nối | | `1.next = 1000000`, `1000004.next = 6` | | |

In [15]:
def trace_merge(list1, a, b, list2):
    before = list1; idx = 0
    print(f"[before] bước 0: chỉ số {idx}, giá trị {before.val}")
    for s in range(a - 1):
        before = before.next; idx += 1
        print(f"[before] bước {s + 1}: chỉ số {idx}, giá trị {before.val}")
    after = before
    for s in range(b - a + 2):
        after = after.next; idx += 1
        print(f"[after ] bước {s + 1}: chỉ số {idx}, giá trị {after.val}")
    tail2 = list2; k = 0
    while tail2.next:
        tail2 = tail2.next; k += 1
    print(f"[tail2 ] sau {k} bước: giá trị {tail2.val}")
    before.next = list2
    tail2.next = after
    print(f"nối: {before.val}.next = {list2.val}, {tail2.val}.next = {after.val}")
    return list1

res = trace_merge(build_list([0, 1, 2, 3, 4, 5, 6]), 2, 5,
                  build_list([1000000, 1000001, 1000002, 1000003, 1000004]))
print("kết quả:", to_list(res))

[before] bước 0: chỉ số 0, giá trị 0
[before] bước 1: chỉ số 1, giá trị 1
[after ] bước 1: chỉ số 2, giá trị 2
[after ] bước 2: chỉ số 3, giá trị 3
[after ] bước 3: chỉ số 4, giá trị 4
[after ] bước 4: chỉ số 5, giá trị 5
[after ] bước 5: chỉ số 6, giá trị 6
[tail2 ] sau 4 bước: giá trị 1000004
nối: 1.next = 1000000, 1000004.next = 6
kết quả: [0, 1, 1000000, 1000001, 1000002, 1000003, 1000004, 6]


## 9. Code nộp LeetCode

In [16]:
class Solution:
    def mergeInBetween(self, list1: ListNode, a: int, b: int, list2: ListNode) -> ListNode:
        before = list1
        for _ in range(a - 1):
            before = before.next
        after = before
        for _ in range(b - a + 2):
            after = after.next
        tail2 = list2
        while tail2.next:
            tail2 = tail2.next
        before.next = list2
        tail2.next = after
        return list1

for l1, a, b, l2, out in examples:
    got = to_list(Solution().mergeInBetween(build_list(l1), a, b, build_list(l2)))
    print(got)
    assert got == out

[10, 1, 13, 1000000, 1000001, 1000002, 5]
[0, 1, 1000000, 1000001, 1000002, 1000003, 1000004, 6]


## 10. Vì sao thuật toán đúng

Gọi các nút của `list1` là `x₀ → x₁ → … → x_{n−1}`, của `list2` là `y₀ → … → y_{m−1}`.

- Vòng 1 đi `a − 1` bước từ `x₀` ⇒ `before = x_{a−1}` (hợp lệ vì `a ≥ 1`).
- Vòng 2 đi thêm `b − a + 2` bước ⇒ `after = x_{a−1+b−a+2} = x_{b+1}` (hợp lệ vì `b + 1 ≤ n − 1`). Vòng này chạy **trước** khi sửa con trỏ nên đi đúng trên `list1` gốc.
- Vòng 3 cho `tail2 = y_{m−1}`.

Sau hai phép gán, đi từ `x₀`: `x₀ → … → x_{a−1}` (không đổi) → `y₀ → … → y_{m−1}` (không đổi) → `x_{b+1} → … → x_{n−1} → None` (không đổi). Đó đúng là `list1[0..a−1] + list2 + list1[b+1..n−1]`. Các nút `x_a..x_b` không còn đường tới từ `x₀` ⇒ đã bị "xóa" khỏi kết quả. Không có chu trình vì mỗi đoạn kết thúc bằng một mối nối đi **tới** đoạn sau và đoạn cuối kết thúc bằng `None`.

## 11. Phân tích Time / Space Complexity

Gọi `n = len(list1)`, `m = len(list2)`.

| Cách | Time | Space phụ |
|---|---|---|
| 1. Sang list Python | `O(n + m)` | `O(n + m)` |
| 2. Hai con trỏ | `O(b + m)` | `O(1)` |
| 3. Một lượt + chỉ số | `O(b + m)` | `O(1)` |
| 4. Dummy tổng quát | `O(b + m)` | `O(1)` |

Không thể nhanh hơn `O(m)` vì phải tìm đuôi `list2` (danh sách đơn không lưu đuôi), và `O(b)` vì phải đi tới vị trí `b+1`. Đo thử với `n = m = 10⁴`:

In [17]:
rnd = random.Random(3)
n = m = 10**4
arr1 = [rnd.randint(-10**9, 10**9) for _ in range(n)]
arr2 = [rnd.randint(-10**9, 10**9) for _ in range(m)]
for name, f in [("baseline", merge_baseline), ("hai con trỏ", SolutionTwoPointers().mergeInBetween),
                ("một lượt", SolutionOnePass().mergeInBetween), ("dummy", SolutionDummy().mergeInBetween)]:
    l1, l2 = build_list(arr1), build_list(arr2)
    t = time.perf_counter(); res = f(l1, 1, n - 2, l2); dt = time.perf_counter() - t
    print(f"{name:12}: {dt * 1000:6.2f} ms, độ dài kết quả {len(to_list(res))}")

baseline    :   5.38 ms, độ dài kết quả 10002
hai con trỏ :   1.40 ms, độ dài kết quả 10002
một lượt    :   1.14 ms, độ dài kết quả 10002
dummy       :   0.58 ms, độ dài kết quả 10002


## 12. Bộ test

Chuẩn đối chiếu: `arr1[:a] + arr2 + arr1[b+1:]`. Ngoài so sánh giá trị, với các cách sửa tại chỗ ta kiểm tra thêm **danh tính nút**: kết quả phải dùng lại đúng các nút gốc (đầu là `list1`, nút sau `before` là `list2`).

In [18]:
INPLACE = [("two", SolutionTwoPointers().mergeInBetween),
           ("one", SolutionOnePass().mergeInBetween),
           ("dummy", SolutionDummy().mergeInBetween),
           ("submit", Solution().mergeInBetween)]
ALL = [("baseline", merge_baseline)] + INPLACE

def check(arr1, a, b, arr2):
    expected = arr1[:a] + arr2 + arr1[b + 1:]
    for name, f in ALL:
        l1, l2 = build_list(arr1), build_list(arr2)
        res = f(l1, a, b, l2)
        assert to_list(res, limit=len(expected) + 5) == expected, (name, arr1, a, b, arr2)
        if name != "baseline":
            assert res is l1 and kth(res, a) is l2, name     # dùng lại nút gốc

# ví dụ đề
for l1, a, b, l2, out in examples:
    check(l1, a, b, l2)
# edge case (tự dựng, đối chiếu bằng slicing)
check([1, 2, 3], 1, 1, [9])                  # n nhỏ nhất, xóa đúng 1 nút
check([1, 2, 3, 4], 1, 2, [9])               # xóa toàn bộ phần giữa
check([5, 5, 5, 5, 5], 2, 2, [5, 5, 5])      # giá trị trùng nhau — phải dựa vào vị trí, không dựa vào giá trị
check(list(range(10)), 1, 8, list(range(100, 150)))
print("ví dụ đề + edge case OK")

ví dụ đề + edge case OK


In [19]:
# ngẫu nhiên, đúng ràng buộc: 3 <= n, 1 <= a <= b < n-1, 1 <= m
rnd = random.Random(2024)
for _ in range(3000):
    n = rnd.randint(3, 40); m = rnd.randint(1, 20)
    a = rnd.randint(1, n - 2); b = rnd.randint(a, n - 2)
    arr1 = [rnd.randint(-5, 5) for _ in range(n)]
    arr2 = [rnd.randint(100, 105) for _ in range(m)]
    check(arr1, a, b, arr2)
# test lớn nhất
arr1 = list(range(10**4)); arr2 = list(range(-10**4, 0))
check(arr1, 1, 10**4 - 2, arr2)
check(arr1, 5000, 5000, arr2)
print("3000 test ngẫu nhiên + test lớn OK")

3000 test ngẫu nhiên + test lớn OK


In [20]:
# Cách 4 (dummy) ngoài ràng buộc đề: a = 0, b = n-1, list2 rỗng — đối chiếu slicing
rnd = random.Random(7)
for _ in range(2000):
    n = rnd.randint(1, 15); m = rnd.randint(0, 6)
    a = rnd.randint(0, n - 1); b = rnd.randint(a, n - 1)
    arr1 = [rnd.randint(0, 9) for _ in range(n)]; arr2 = [rnd.randint(50, 59) for _ in range(m)]
    res = SolutionDummy().mergeInBetween(build_list(arr1), a, b, build_list(arr2))
    assert to_list(res) == arr1[:a] + arr2 + arr1[b + 1:]
print("Cách 4 tổng quát OK")

Cách 4 tổng quát OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Đi `a` bước thay vì `a − 1` bước (lệch một)

In [21]:
def wrong_off_by_one(list1, a, b, list2):
    before = list1
    for _ in range(a):                 # SAI: tới chỉ số a (nút đầu tiên cần xóa)
        before = before.next
    after = list1
    for _ in range(b + 1):
        after = after.next
    tail2 = list2
    while tail2.next: tail2 = tail2.next
    before.next, tail2.next = list2, after
    return list1

l1, a, b, l2, out = examples[0]
print("sai :", to_list(wrong_off_by_one(build_list(l1), a, b, build_list(l2))))
print("đúng:", out)

sai : [10, 1, 13, 6, 1000000, 1000001, 1000002, 5]
đúng: [10, 1, 13, 1000000, 1000001, 1000002, 5]


**Vì sao sai:** `before` phải là nút **đứng trước** đoạn bị xóa (chỉ số `a−1`). Đi `a` bước làm nút `6` (chỉ số 3) vẫn bị giữ lại. **Sửa:** `range(a - 1)`, hoặc dùng `dummy` và `range(a)`.

### Lỗi 2 — Gán `before.next = list2` rồi mới đi tìm `after` từ `before`

In [22]:
def wrong_order(list1, a, b, list2):
    before = list1
    for _ in range(a - 1):
        before = before.next
    before.next = list2                # nối trước ...
    after = before
    for _ in range(b - a + 2):         # ... SAI: giờ đang đi trên list2!
        after = after.next if after else None
    tail2 = list2
    while tail2.next: tail2 = tail2.next
    tail2.next = after
    return list1

l1, a, b, l2, out = examples[1]
print("sai :", to_list(wrong_order(build_list(l1), a, b, build_list(l2)), limit=30))
print("đúng:", out)

sai : [0, 1, 1000000, 1000001, 1000002, 1000003, 1000004, 1000004, 1000004, 1000004, 1000004, 1000004, 1000004, 1000004, 1000004, 1000004, 1000004, 1000004, 1000004, 1000004, 1000004, 1000004, 1000004, 1000004, 1000004, 1000004, 1000004, 1000004, 1000004, 1000004]
đúng: [0, 1, 1000000, 1000001, 1000002, 1000003, 1000004, 6]


**Vì sao sai:** sau `before.next = list2`, đi tiếp từ `before` là đi vào `list2`, không còn tới được phần sau của `list1` ⇒ `after` sai (ở đây trỏ vào giữa `list2`, tạo ra chu trình — `to_list` phải dùng `limit` để không treo). **Sửa:** tìm **xong** `before`, `after`, `tail2` rồi mới gán.

### Lỗi 3 — Quên tìm đuôi `list2`, nối thẳng `list2.next = after`

In [23]:
def wrong_no_tail(list1, a, b, list2):
    before = list1
    for _ in range(a - 1):
        before = before.next
    after = before
    for _ in range(b - a + 2):
        after = after.next
    before.next = list2
    list2.next = after                 # SAI: mất các nút còn lại của list2
    return list1

l1, a, b, l2, out = examples[0]
print("sai :", to_list(wrong_no_tail(build_list(l1), a, b, build_list(l2))))
print("đúng:", out)

sai : [10, 1, 13, 1000000, 5]
đúng: [10, 1, 13, 1000000, 1000001, 1000002, 5]


**Vì sao sai:** chỉ nút **đầu** của `list2` được nối, phần còn lại bị cắt mất. **Sửa:** đi tới `tail2` (nút có `next is None`) rồi `tail2.next = after`.

### Lỗi 4 — Tìm điểm cắt theo **giá trị** thay vì theo **vị trí**

In [24]:
def wrong_by_value(list1, a, b, list2):
    arr = to_list(list1)
    va, vb = arr[a], arr[b]
    i = arr.index(va); j = arr.index(vb)    # SAI: index() trả vị trí xuất hiện ĐẦU TIÊN
    return build_list(arr[:i] + to_list(list2) + arr[j + 1:])

arr1, a, b, arr2 = [1, 7, 3, 7, 7, 2], 3, 4, [0]
print("sai :", to_list(wrong_by_value(build_list(arr1), a, b, build_list(arr2))))
print("đúng:", arr1[:a] + arr2 + arr1[b + 1:])

sai : [1, 0, 3, 7, 7, 2]
đúng: [1, 7, 3, 0, 2]


**Vì sao sai:** đề nói nút thứ `a` tới thứ `b` — là **vị trí**. Giá trị có thể trùng nhau (ràng buộc không cấm), `index` trả vị trí đầu tiên ⇒ cắt nhầm chỗ. **Sửa:** đếm bước.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Bài linked list "xóa/chèn/thay một đoạn" ⇒ luôn vẽ **3 đoạn + 2 mối nối**, xác định nút đứng **trước** và **sau** đoạn bị thay.
- Muốn tới nút "đứng trước chỉ số `k`" ⇒ đi `k − 1` bước từ `head`, hoặc `k` bước từ `dummy`. Dummy giúp không phải xét riêng `k = 0`.
- Thu thập **mọi tham chiếu cần dùng trước**, gán `next` **sau cùng** — tránh lạc đường.
- Khi debug, viết `to_list(head, limit=...)` có giới hạn để phát hiện chu trình thay vì treo kernel.
- Test với giá trị **trùng nhau** để chắc chắn code dựa vào vị trí.

## 15. Biến thể của bài

1. **LC 203 — Remove Linked List Elements / LC 19 — Remove Nth Node From End:** xóa nút, dùng dummy.
2. **LC 92 — Reverse Linked List II:** cùng việc tìm nút trước/sau một đoạn `[left, right]`, nhưng đảo đoạn giữa.
3. **LC 25 — Reverse Nodes in k-Group:** lặp lại thao tác "cắt đoạn – xử lý – nối lại".
4. **LC 2095 — Delete the Middle Node:** tìm nút trước điểm giữa.
5. **Chèn không xóa:** chèn `list2` vào **sau** nút chỉ số `k` (bài tập 5).
6. **Trả cả đoạn bị xóa** thành một danh sách riêng, kết thúc bằng `None` (bài tập 4).

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. List Python | cắt-ghép slicing, dựng lại | `O(n+m)` | `O(n+m)` | kiểm chứng, khi không cần giữ nút gốc |
| 2. Hai con trỏ | đếm bước tới `a−1` và `b+1` | `O(b+m)` | `O(1)` | **nộp bài** |
| 3. Một lượt + chỉ số | đánh dấu mốc khi duyệt | `O(b+m)` | `O(1)` | nhiều mốc, dễ đọc |
| 4. Dummy tổng quát | dummy + `after` có thể None | `O(b+m)` | `O(1)` | biến thể cho phép `a = 0`/`b = n−1`/list2 rỗng |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** (a) `list1 = [1,2,3,4,5]`, `a = 1`, `b = 3`, `list2 = [9]`; (b) `list1 = [1,2,3]`, `a = b = 1`, `list2 = [7,8,9]`.

**Bài 2 (tính tay):** Với `n = 6`, `a = 3`, `b = 4` (ví dụ 1) và `m = 3`, Cách 2 thực hiện tổng cộng bao nhiêu phép `= .next`?

**Bài 3 (điền TODO):** Hoàn thiện `node_before(head, k)` trả nút ở chỉ số `k − 1` (giả sử `k ≥ 1`).

**Bài 4 (tự viết lại):** Viết `merge_and_return_removed(list1, a, b, list2)` trả về **cặp** `(đầu kết quả, đầu đoạn bị xóa)`, trong đó đoạn bị xóa là danh sách độc lập kết thúc bằng `None`.

**Bài 5 (mini challenge):** Viết `insert_after(list1, k, list2)` chèn **toàn bộ** `list2` vào **sau** nút chỉ số `k` của `list1` (không xóa gì), cho phép `k = -1` nghĩa là chèn vào đầu và `list2` có thể rỗng.

In [25]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def node_before(head, k):
    cur = head
    # TODO: đi đúng số bước
    return cur

In [26]:
# Bài 4, 5 — khung tự viết (chỉ định nghĩa, không gọi)
def merge_and_return_removed(list1, a, b, list2):
    # TODO
    pass

def insert_after(list1, k, list2):
    # TODO
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** Mức 1: giữ `arr1[:a]` và `arr1[b+1:]`. Mức 2: (a) giữ `[1]` và `[5]`. Mức 3: (a) `[1,9,5]`; (b) `[1,7,8,9,3]`.

**Bài 2.** Mức 1: vòng 1 có `a − 1` bước. Mức 2: vòng 2 có `b − a + 2` bước, vòng 3 có `m − 1` bước. Mức 3: `2 + 3 + 2 = 7`.

**Bài 3.** Mức 1: từ `head` (chỉ số 0). Mức 2: cần tới chỉ số `k − 1`. Mức 3: `for _ in range(k - 1): cur = cur.next`.

**Bài 4.** Mức 1: lưu `first_removed = before.next` trước khi nối. Mức 2: nút chỉ số `b` là `last_removed`. Mức 3: `last_removed.next = None` sau khi đã có `after`.

**Bài 5.** Mức 1: dùng dummy để `k = -1` thành "sau dummy". Mức 2: nếu `list2` rỗng trả nguyên `list1`. Mức 3: `nxt = prev.next; prev.next = list2; tail2.next = nxt`.

## 19. Đáp án tham khảo cho bài tập

In [27]:
# Bài 1
b1a = to_list(Solution().mergeInBetween(build_list([1, 2, 3, 4, 5]), 1, 3, build_list([9])))
b1b = to_list(Solution().mergeInBetween(build_list([1, 2, 3]), 1, 1, build_list([7, 8, 9])))
print("Bài 1:", b1a, b1b)
assert b1a == [1, 9, 5] and b1b == [1, 7, 8, 9, 3]

# Bài 2 — đếm phép .next bằng cách đếm số vòng lặp
def count_steps(n, a, b, m):
    return (a - 1) + (b - a + 2) + (m - 1)
print("Bài 2:", count_steps(6, 3, 4, 3))
assert count_steps(6, 3, 4, 3) == 7

Bài 1: [1, 9, 5] [1, 7, 8, 9, 3]
Bài 2: 7


In [28]:
# Bài 3
def node_before_sol(head, k):
    cur = head
    for _ in range(k - 1):
        cur = cur.next
    return cur

h = build_list([10, 1, 13, 6, 9, 5])
assert [node_before_sol(h, k).val for k in range(1, 6)] == [10, 1, 13, 6, 9]

# Bài 4
def merge_and_return_removed_sol(list1, a, b, list2):
    before = node_before_sol(list1, a)
    first_removed = before.next
    last_removed = first_removed
    for _ in range(b - a):
        last_removed = last_removed.next
    after = last_removed.next
    tail2 = list2
    while tail2.next:
        tail2 = tail2.next
    before.next = list2
    tail2.next = after
    last_removed.next = None
    return list1, first_removed

rnd = random.Random(41)
for _ in range(1500):
    n = rnd.randint(3, 30); m = rnd.randint(1, 10)
    a = rnd.randint(1, n - 2); b = rnd.randint(a, n - 2)
    arr1 = [rnd.randint(0, 9) for _ in range(n)]; arr2 = [rnd.randint(0, 9) for _ in range(m)]
    res, rem = merge_and_return_removed_sol(build_list(arr1), a, b, build_list(arr2))
    assert to_list(res) == arr1[:a] + arr2 + arr1[b + 1:]
    assert to_list(rem) == arr1[a:b + 1]
print("Bài 3, 4 OK")

Bài 3, 4 OK


In [29]:
# Bài 5
def insert_after_sol(list1, k, list2):
    if list2 is None:
        return list1
    dummy = ListNode(0, list1)
    prev = dummy
    for _ in range(k + 1):              # dummy là chỉ số -1
        prev = prev.next
    tail2 = list2
    while tail2.next:
        tail2 = tail2.next
    nxt = prev.next
    prev.next = list2
    tail2.next = nxt
    return dummy.next

rnd = random.Random(42)
for _ in range(2000):
    n = rnd.randint(0, 12); m = rnd.randint(0, 5)
    k = rnd.randint(-1, n - 1)
    arr1 = [rnd.randint(0, 9) for _ in range(n)]; arr2 = [rnd.randint(50, 59) for _ in range(m)]
    res = insert_after_sol(build_list(arr1), k, build_list(arr2))
    assert to_list(res) == arr1[:k + 1] + arr2 + arr1[k + 1:]
print("Bài 5 OK, ví dụ:", to_list(insert_after_sol(build_list([1, 2, 3]), -1, build_list([0]))))

Bài 5 OK, ví dụ: [0, 1, 2, 3]


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template "thay đoạn [a, b] của list1 bằng list2":**

```python
before = list1
for _ in range(a - 1):          # chỉ số a-1
    before = before.next
after = before
for _ in range(b - a + 2):      # chỉ số b+1
    after = after.next
tail2 = list2
while tail2.next:               # đuôi list2
    tail2 = tail2.next
before.next = list2             # mối nối 1
tail2.next = after              # mối nối 2
return list1
```

**Biến thể có dummy (cho phép a = 0):** `before = dummy; for _ in range(a): before = before.next` và trả `dummy.next`.

**Checklist:**
- [ ] `before` ở chỉ số `a − 1`, `after` ở chỉ số `b + 1` (tự đếm lại trên ví dụ 1)?
- [ ] Đã tìm `after` **trước** khi gán `before.next = list2`?
- [ ] Đã đi tới **đuôi** `list2` (không nối thẳng từ đầu)?
- [ ] Dựa vào **vị trí**, không dựa vào giá trị?
- [ ] Trả về `list1` (đầu không đổi vì `a ≥ 1`)?
- [ ] Đã đối chiếu với `arr1[:a] + arr2 + arr1[b+1:]` trên dữ liệu ngẫu nhiên?